# Python: Introdução ao Biopython

**Biopython** é um conjunto de ferramentas Python de código aberto para bioinformática. Ele fornece classes e funções para manipular sequências de DNA, RNA e proteína, ler e escrever vários formatos de arquivo biológicos (como FASTA, GenBank, PDB), interagir com bancos de dados biológicos online (como NCBI), e muito mais. Se você trabalha com biologia computacional, o Biopython é uma biblioteca indispensável.

## 1. Instalação do Biopython

Antes de usar o Biopython, você precisa instalá-lo. No Google Colab, você pode fazer isso usando o `pip` diretamente em uma célula de código.

In [ ]:
# Instala o Biopython
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 45.6 MB/s eta 0:00:00


## 2. Manipulação Básica de Sequências com `Bio.Seq`

O módulo `Bio.Seq` fornece a classe `Seq` para representar sequências biológicas. Ele permite realizar operações como transcrição, tradução, complemento reverso, e muito mais.

In [ ]:
from Bio.Seq import Seq

# Criando uma sequência de DNA
sequencia_dna = Seq("ATGCGTACGTAGCTAGCATGC")
print(f"Sequência DNA: {sequencia_dna}")

# Obtendo o complemento reverso
complemento_reverso = sequencia_dna.reverse_complement()
print(f"Complemento Reverso: {complemento_reverso}")

# Transcrevendo DNA para RNA
sequencia_rna = sequencia_dna.transcribe()
print(f"Sequência RNA: {sequencia_rna}")

# Traduzindo RNA para proteína (com códons de parada '*' inclusos)
# Note que a tradução padrão de uma sequência de DNA vai primeiro transcrevê-la para RNA
# e depois traduzi-la. Se a sequência for RNA, use Seq('...', alphabet=RNAAlphabet()).translate()
sequencia_proteina = sequencia_dna.translate()
print(f"Sequência Proteica (DNA -> RNA -> Proteína): {sequencia_proteina}")

# Traduzindo uma sequência de RNA (após a transcrição)
sequencia_proteina_rna = sequencia_rna.translate()
print(f"Sequência Proteica (RNA -> Proteína): {sequencia_proteina_rna}")

# Calculando o conteúdo GC manualmente (para contornar problema de importação de Bio.SeqUtils.GC_content)
gc_count = sequencia_dna.count('G') + sequencia_dna.count('C')
gc_content = (gc_count / len(sequencia_dna)) * 100
print(f"Conteúdo GC (calculado manualmente): {gc_content:.2f}%")

Sequência DNA: ATGCGTACGTAGCTAGCATGC
Complemento Reverso: GCATGCTAGCTACGTACGCAT
Sequência RNA: AUGCGUACGUAGCUAGCAUGC
Sequência Proteica (DNA -> RNA -> Proteína): MRT*LAC
Sequência Proteica (RNA -> Proteína): MRT*LAC
Conteúdo GC (calculado manualmente): 52.38%


## 3. Lendo e Escrevendo Arquivos de Sequência com `Bio.SeqIO`

`Bio.SeqIO` é a interface principal do Biopython para trabalhar com arquivos de sequência em vários formatos, como FASTA, GenBank, FASTQ, etc. Ele permite analisar (parse) registros de sequência e escrevê-los de volta.

In [ ]:
from Bio import SeqIO

# --- Criando um arquivo FASTA de exemplo ---
# No Colab, você cria o arquivo diretamente para que o Biopython possa lê-lo.
fasta_content = (
    ">gene_a | Exemplo de Gene A\n"
    "ATGGCCATGCGTACGTAGCTAGCATGC\n"
    "TTAGCTACGTAGCATGC\n"
    ">gene_b | Exemplo de Gene B\n"
    "ATGAAAAAAGGGGGGCCCCTTTAA\n"
    "AAAAAGGGGGGCCCCTTTAA\n"
)

with open("exemplo.fasta", "w") as f:
    f.write(fasta_content)
print("Arquivo 'exemplo.fasta' criado.")

# --- Lendo um arquivo FASTA ---
print("\n--- Lendo o arquivo FASTA ---")
for record in SeqIO.parse("exemplo.fasta", "fasta"):
    print(f"ID: {record.id}")
    print(f"Nome: {record.name}")
    print(f"Descrição: {record.description}")
    print(f"Sequência: {record.seq}")
    print(f"Comprimento: {len(record.seq)}\n")

# --- Escrevendo (filtrando) sequências para um novo arquivo FASTA ---
# Vamos escrever apenas as sequências com comprimento > 20
records_to_write = []
for record in SeqIO.parse("exemplo.fasta", "fasta"):
    if len(record.seq) > 20:
        records_to_write.append(record)

with open("sequencias_longas.fasta", "w") as output_handle:
    SeqIO.write(records_to_write, output_handle, "fasta")
print("Arquivo 'sequencias_longas.fasta' criado com sequências mais longas.")

# Verificando o conteúdo do novo arquivo
print("\n--- Conteúdo de 'sequencias_longas.fasta' ---")
with open("sequencias_longas.fasta", "r") as f:
    print(f.read())

Arquivo 'exemplo.fasta' criado.

--- Lendo o arquivo FASTA ---
ID: gene_a
Nome: gene_a
Descrição: gene_a | Exemplo de Gene A
Sequência: ATGGCCATGCGTACGTAGCTAGCATGCTTAGCTACGTAGCATGC
Comprimento: 44

ID: gene_b
Nome: gene_b
Descrição: gene_b | Exemplo de Gene B
Sequência: ATGAAAAAAGGGGGGCCCCTTTAAAAAAAGGGGGGCCCCTTTAA
Comprimento: 44

Arquivo 'sequencias_longas.fasta' criado com sequências mais longas.

--- Conteúdo de 'sequencias_longas.fasta' ---
>gene_a | Exemplo de Gene A
ATGGCCATGCGTACGTAGCTAGCATGCTTAGCTACGTAGCATGC
>gene_b | Exemplo de Gene B
ATGAAAAAAGGGGGGCCCCTTTAAAAAAAGGGGGGCCCCTTTAA



## 4. Acessando Bancos de Dados Biológicos com `Bio.Entrez`

`Bio.Entrez` permite interagir com os bancos de dados do NCBI (National Center for Biotechnology Information), como PubMed, GenBank, GEO, etc., para buscar e baixar dados programaticamente.

In [ ]:
from Bio import Entrez

# É obrigatório fornecer seu endereço de e-mail ao usar o Entrez
Entrez.email = "seu.email@exemplo.com" # Substitua pelo seu e-mail real

# --- Buscando IDs no banco de dados PubMed ---
print("\n--- Buscando artigos no PubMed ---")
handle = Entrez.esearch(db="pubmed", term="CRISPR gene editing", retmax="3")
record = Entrez.read(handle)
handle.close()
print(f"IDs de artigos encontrados: {record['IdList']}")

# --- Baixando um registro do banco de dados Nucleotide (GenBank) ---
# Vamos usar um ID de exemplo para uma sequência de gene
genbank_id = "NM_000546" # Exemplo: mRNA humano TP53

print(f"\n--- Baixando o registro {genbank_id} do GenBank ---")
handle = Entrez.efetch(db="nucleotide", id=genbank_id, rettype="gb", retmode="text")
genbank_record = handle.read()
handle.close()
# print(genbank_record[:500]) # Imprime as primeiras 500 caracteres do registro GenBank
print(f"Registro {genbank_id} baixado (parcialmente exibido):\n{genbank_record[:500]}...\n")

# Você pode analisar este registro GenBank com SeqIO.read() também
# handle = Entrez.efetch(db="nucleotide", id=genbank_id, rettype="gb", retmode="text")
# record_gb = SeqIO.read(handle, "genbank")
# handle.close()
# print(f"ID do registro GenBank: {record_gb.id}")
# print(f"Sequência do registro GenBank: {record_gb.seq[:50]}...")


--- Buscando artigos no PubMed ---
IDs de artigos encontrados: ['41885989', '41885988', '41885675']

--- Baixando o registro NM_000546 do GenBank ---
Registro NM_000546 baixado (parcialmente exibido):
LOCUS       NM_000546               2512 bp    mRNA    linear   PRI 21-NOV-2025
DEFINITION  Homo sapiens tumor protein p53 (TP53), transcript variant 1, mRNA.
ACCESSION   NM_000546
VERSION     NM_000546.6
KEYWORDS    RefSeq; MANE Select.
SOURCE      Homo sapiens (human)
  ORGANISM  Homo sapiens
            Eukaryota; Metazoa; Chordata; Craniata; Vertebrata; Euteleostomi;
            Mammalia; Eutheria; Euarchontoglires; Primates; Haplorrhini;
            Catarrhini; Hominidae; Homo.
REFERENCE   ...



## 5. Análise Estrutural com `Bio.PDB`

`Bio.PDB` é um módulo poderoso no Biopython para trabalhar com estruturas de proteínas, como aquelas encontradas em arquivos PDB (Protein Data Bank). Ele permite analisar modelos, cadeias, resíduos e átomos de uma estrutura molecular.

In [ ]:
from Bio.PDB import PDBList, PDBParser

# Baixando um arquivo PDB de exemplo (Hemoglobina, ID: 1hba)
pdb_list = PDBList()
pdb_file = pdb_list.retrieve_pdb_file('1hba', pdir='.', file_format='pdb')
print(f"Arquivo PDB baixado: {pdb_file}")

# Criando um parser PDB
parser = PDBParser()

# Analisando a estrutura
structure = parser.get_structure('1hba', pdb_file)

print(f"\nAnálise da estrutura '1hba':")

# Iterando sobre os modelos (geralmente há apenas um)
for model in structure:
    print(f"  Modelo ID: {model.id}")
    print(f"  Número de cadeias: {len(model)}")
    for chain in model:
        print(f"    Cadeia ID: {chain.id}, Número de resíduos: {len(chain)}")
        # Exemplo de acesso a resíduos e átomos
        # for i, residue in enumerate(chain):
        #     if i < 5: # Apenas os primeiros 5 para exemplo
        #         print(f"      Resíduo: {residue.get_resname()} {residue.id[1]}, Átomos: {len(residue)}")

# Você pode fazer análises mais complexas, como calcular distâncias, ângulos, etc.
# Por exemplo, para obter o nome da primeira cadeia:
first_chain = structure[0]['A']
print(f"\nPrimeira cadeia (A) do primeiro modelo: {first_chain.id}")

Arquivo PDB baixado: ./pdb1hba.ent

Análise da estrutura '1hba':
  Modelo ID: 0
  Número de cadeias: 4
    Cadeia ID: A, Número de resíduos: 205
    Cadeia ID: B, Número de resíduos: 201
    Cadeia ID: C, Número de resíduos: 183
    Cadeia ID: D, Número de resíduos: 192

Primeira cadeia (A) do primeiro modelo: A


/usr/local/lib/python3.12/dist-packages/Bio/PDB/StructureBuilder.py:100: PDBConstructionWarning: WARNING: Chain A is discontinuous at line 4943.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/Bio/PDB/StructureBuilder.py:100: PDBConstructionWarning: WARNING: Chain B is discontinuous at line 4986.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/Bio/PDB/StructureBuilder.py:100: PDBConstructionWarning: WARNING: Chain C is discontinuous at line 5029.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/Bio/PDB/StructureBuilder.py:100: PDBConstructionWarning: WARNING: Chain D is discontinuous at line 5072.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/Bio/PDB/StructureBuilder.py:100: PDBConstructionWarning: WARNING: Chain A is discontinuous at line 5117.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/Bio/PDB/StructureBuilder.py:100: PDBConstructionWarning: WARNING: Chain B is discontinuous at line 5180.
  warnings.warn(
/usr/local/lib/python3.12/di

## Conclusão

O Biopython é uma ferramenta incrivelmente poderosa para qualquer biólogo ou bioinformacionista que trabalha com Python. Ele simplifica tarefas complexas de manipulação de sequências, processamento de arquivos biológicos e interação com grandes bancos de dados. Dominar o Biopython abrirá um leque de possibilidades para suas análises e projetos de bioinformática, permitindo que você escreva código mais eficiente, robusto e escalável para lidar com os desafios do mundo biológico.